# EXP-09: Stacking Ensemble — BERTimbau + SVC + LightGBM

**Competition:** spr-2026-mammography-report-classification

**Architecture:**
- **Level 0 — BERTimbau Base** with FGM adversarial training, weighted Focal Loss, multi-sample dropout, cosine schedule
- **Level 0 — TF-IDF (word + char) + CalibratedLinearSVC**
- **Level 0 — LightGBM** with TF-IDF + 30 hand-crafted clinical features
- **Level 1 — Ridge meta-learner** stacking on OOF probabilities

**Novelty:** FGM adversarial perturbation on BERT embeddings, multi-sample dropout head, stacking (not simple weighted average).

**Setup:** Add datasets `cortese/bert-base-portuguese-cased` and `spr-2026-mammography-report-classification` to notebook inputs.

In [ ]:
import os, gc, re, glob, time, hashlib, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifier
from scipy.sparse import hstack, csr_matrix
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    get_cosine_schedule_with_warmup,
)

warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
    HAS_LGB = True
    print(f'LightGBM {lgb.__version__}')
except ImportError:
    !pip install -q lightgbm
    import lightgbm as lgb
    HAS_LGB = True
    print(f'LightGBM {lgb.__version__}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# === Config ===
def find_input(pattern):
    for root, dirs, files in os.walk('/kaggle/input'):
        for d in dirs:
            if pattern in d.lower():
                return os.path.join(root, d)
    hits = glob.glob(f'/kaggle/input/**/*{pattern}*', recursive=True)
    return hits[0] if hits else None

MODEL_PATH = find_input('bert-base-portuguese-cased')
COMP_DIR   = find_input('spr-2026-mammography')

assert MODEL_PATH, f'Model dataset not found! {os.listdir("/kaggle/input")}'
assert COMP_DIR,   f'Competition not found! {os.listdir("/kaggle/input")}'

# --- BERT config ---
NUM_LABELS   = 7
MAX_LEN      = 256
BERT_BS      = 32
BERT_EPOCHS  = 5
BERT_LR      = 2e-5
WARMUP_RATIO = 0.1
WD           = 0.01
FOCAL_GAMMA  = 2.0
FGM_EPS      = 0.5          # adversarial perturbation epsilon
MULTI_DROP_N = 5             # multi-sample dropout passes
MULTI_DROP_P = 0.3           # dropout probability

# --- CV config ---
N_SPLITS = 5
SEED     = 42

# --- LightGBM config ---
LGB_PARAMS = dict(
    objective='multiclass', num_class=NUM_LABELS, metric='multi_logloss',
    learning_rate=0.05, num_leaves=63, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=5, class_weight='balanced',
    verbose=-1, n_jobs=-1, seed=SEED,
)
LGB_ROUNDS   = 1500
LGB_EARLY    = 50

print(f'MODEL_PATH: {MODEL_PATH}')
print(f'COMP_DIR:   {COMP_DIR}')

In [ ]:
# === Load Data ===
train_df = pd.read_csv(os.path.join(COMP_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(COMP_DIR, 'test.csv'))
print(f'Train: {train_df.shape} | Test: {test_df.shape}')
print(train_df['target'].value_counts().sort_index())

In [ ]:
# === Text Cleaning ===
def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower().strip()
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{2,}', '\n', text)
    return text

train_df['clean'] = train_df['report'].apply(clean_text)
test_df['clean']  = test_df['report'].apply(clean_text)

def stable_hash(s):
    return hashlib.md5(s.encode('utf-8')).hexdigest()

groups = train_df['clean'].apply(stable_hash).values
labels = train_df['target'].values

print(f'Unique groups: {len(set(groups))} / {len(groups)}')

In [ ]:
# === Clinical Feature Engineering (30 features) ===

def extract_clinical_features(df, col='clean'):
    """Extract hand-crafted clinical features from mammography reports."""
    texts = df[col].values
    feats = pd.DataFrame(index=df.index)

    # --- Length features ---
    feats['n_chars']     = [len(t) for t in texts]
    feats['n_words']     = [len(t.split()) for t in texts]
    feats['n_lines']     = [t.count('\n') + 1 for t in texts]
    feats['n_sentences'] = [len(re.split(r'[.!?]+', t)) for t in texts]

    # --- Section presence ---
    feats['has_indicacao']   = [int('indicaç' in t or 'indicacao' in t) for t in texts]
    feats['has_achados']     = [int('achados' in t or 'achado' in t) for t in texts]
    feats['has_comparativa'] = [int('comparati' in t or 'estudo anterior' in t) for t in texts]
    feats['has_conclusao']   = [int('conclus' in t or 'opinião' in t or 'opiniao' in t) for t in texts]

    # --- Indication type ---
    feats['is_screening']     = [int('rastreamento' in t or 'rastreio' in t or 'rotina' in t) for t in texts]
    feats['is_reevaluation']  = [int('reavaliação' in t or 'reavaliacao' in t or 'controle' in t) for t in texts]
    feats['is_followup']      = [int('acompanhamento' in t or 'seguimento' in t) for t in texts]

    # --- Benign indicators ---
    feats['has_benignas']          = [int('benigna' in t or 'benigno' in t) for t in texts]
    feats['has_lipossubstituidas'] = [int('lipossubstitu' in t or 'liposubstitu' in t) for t in texts]
    feats['has_no_suspicious']     = [int('sem alteraç' in t or 'sem alteracao' in t or 'exame normal' in t) for t in texts]
    feats['has_cisto']             = [int('cisto' in t or 'cístico' in t) for t in texts]

    # --- Suspicious findings ---
    feats['has_nodulo']       = [int('nódulo' in t or 'nodulo' in t) for t in texts]
    feats['has_espiculado']   = [int('espiculad' in t) for t in texts]
    feats['has_pleomorficas'] = [int('pleomórf' in t or 'pleomorf' in t) for t in texts]
    feats['has_distorcao']    = [int('distorção' in t or 'distorcao' in t or 'distorsão' in t) for t in texts]
    feats['has_assimetria']   = [int('assimetria' in t or 'asimetria' in t) for t in texts]
    feats['has_irregular']    = [int('irregular' in t) for t in texts]
    feats['has_microcalc']    = [int('microcalcific' in t) for t in texts]

    # --- High suspicion / malignancy ---
    feats['has_biopsia']     = [int('biópsia' in t or 'biopsia' in t) for t in texts]
    feats['has_cancer']      = [int('câncer' in t or 'cancer' in t or 'carcinoma' in t) for t in texts]
    feats['has_malignidade'] = [int('maligno' in t or 'malignidade' in t or 'maligna' in t) for t in texts]
    feats['has_retracao']    = [int('retração' in t or 'retracao' in t) for t in texts]

    # --- Additional imaging ---
    feats['has_us_complement'] = [int('ultrassonogra' in t or 'ecografia' in t or 'complemento' in t) for t in texts]
    feats['has_additional']    = [int('incidência adicional' in t or 'incidencia adicional' in t or 'compressão' in t) for t in texts]

    # --- Composite risk score ---
    feats['risk_score'] = (
        - 3 * feats['has_no_suspicious']
        - 2 * feats['has_lipossubstituidas']
        - 1 * feats['has_benignas']
        + 1 * feats['has_nodulo']
        + 2 * feats['has_assimetria']
        + 2 * feats['has_microcalc']
        + 3 * feats['has_espiculado']
        + 3 * feats['has_irregular']
        + 3 * feats['has_distorcao']
        + 4 * feats['has_cancer']
        + 4 * feats['has_malignidade']
        + 4 * feats['has_biopsia']
    )

    return feats

train_clinical = extract_clinical_features(train_df)
test_clinical  = extract_clinical_features(test_df)
CLINICAL_COLS  = train_clinical.columns.tolist()
print(f'Clinical features: {len(CLINICAL_COLS)}')
print(train_clinical.describe().T[['mean','std','min','max']])

---
## Level 0 — Model A: BERTimbau + FGM + Multi-Sample Dropout

In [ ]:
# === BERT Dataset & Model ===

class ReportDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


class MultiSampleDropoutHead(nn.Module):
    """Classification head with multi-sample dropout for regularization.
    At training time, runs N forward passes with different dropout masks
    and averages the loss — reduces overfitting and improves calibration."""

    def __init__(self, hidden_size, num_labels, n_drops=5, drop_p=0.3):
        super().__init__()
        self.dropouts = nn.ModuleList([nn.Dropout(drop_p) for _ in range(n_drops)])
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, hidden_state):
        if self.training:
            logits = torch.stack(
                [self.classifier(drop(hidden_state)) for drop in self.dropouts]
            )
            return logits.mean(dim=0)
        else:
            return self.classifier(hidden_state)


class BERTClassifier(nn.Module):
    def __init__(self, model_path, num_labels, n_drops=5, drop_p=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_path)
        hidden = self.bert.config.hidden_size
        self.head = MultiSampleDropoutHead(hidden, num_labels, n_drops, drop_p)

    def forward(self, input_ids, attention_mask, token_type_ids=None, **kwargs):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                        token_type_ids=token_type_ids)
        cls = out.last_hidden_state[:, 0]  # [CLS] token
        return self.head(cls)


class FGM:
    """Fast Gradient Method — adversarial perturbation on embeddings.
    Adds small perturbation to word embeddings during training to make
    model more robust. Common Kaggle NLP trick that typically adds +0.5-1%."""

    def __init__(self, model, eps=0.5, emb_name='word_embeddings'):
        self.model = model
        self.eps = eps
        self.emb_name = emb_name
        self.backup = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0:
                    param.data.add_(self.eps * param.grad / norm)

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


class WeightedFocalLoss(nn.Module):
    """Focal Loss with per-class alpha weights."""
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.register_buffer('alpha', alpha)
        self.gamma = gamma

    def forward(self, logits, labels):
        ce = F.cross_entropy(logits, labels, reduction='none')
        pt = torch.exp(-ce)
        alpha_t = self.alpha[labels]
        loss = alpha_t * ((1 - pt) ** self.gamma) * ce
        return loss.mean()


def compute_class_weights(labels, num_classes=7):
    """Inverse frequency class weights, normalized."""
    counts = np.bincount(labels, minlength=num_classes).astype(float)
    weights = len(labels) / (num_classes * counts + 1e-6)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)

print('BERT components defined.')

In [ ]:
# === BERT Training Loop with FGM ===

def train_bert_fold(tr_idx, va_idx, fold):
    """Train one fold of BERTimbau and return OOF + test probabilities."""
    print(f'\n{"="*40} BERT FOLD {fold} {"="*40}')

    tr_texts  = train_df.iloc[tr_idx]['report'].tolist()
    va_texts  = train_df.iloc[va_idx]['report'].tolist()
    tr_labels = labels[tr_idx]
    va_labels = labels[va_idx]

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    tr_enc = tokenizer(tr_texts, truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
    va_enc = tokenizer(va_texts, truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')

    tr_ds = ReportDataset(tr_enc, tr_labels)
    va_ds = ReportDataset(va_enc)

    tr_loader = DataLoader(tr_ds, batch_size=BERT_BS, shuffle=True, num_workers=2, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    model = BERTClassifier(MODEL_PATH, NUM_LABELS, MULTI_DROP_N, MULTI_DROP_P).to(DEVICE)

    # Class-weighted focal loss
    alpha = compute_class_weights(tr_labels).to(DEVICE)
    criterion = WeightedFocalLoss(alpha, gamma=FOCAL_GAMMA)

    # Optimizer + Scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=BERT_LR, weight_decay=WD)
    total_steps = len(tr_loader) * BERT_EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    fgm = FGM(model, eps=FGM_EPS)
    scaler = torch.cuda.amp.GradScaler()

    best_f1 = 0
    best_oof_probs = None
    best_state = None

    for epoch in range(BERT_EPOCHS):
        model.train()
        total_loss = 0
        for batch in tr_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            token_type_ids = batch.get('token_type_ids', torch.zeros_like(input_ids)).to(DEVICE)
            lbl = batch['labels'].to(DEVICE)

            # --- Forward pass ---
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                logits = model(input_ids, attention_mask, token_type_ids)
                loss = criterion(logits, lbl)
            scaler.scale(loss).backward()

            # --- FGM adversarial step ---
            fgm.attack()
            with torch.cuda.amp.autocast():
                logits_adv = model(input_ids, attention_mask, token_type_ids)
                loss_adv = criterion(logits_adv, lbl)
            scaler.scale(loss_adv).backward()
            fgm.restore()

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()

        # --- Validation ---
        model.eval()
        all_probs = []
        with torch.no_grad():
            for batch in va_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                token_type_ids = batch.get('token_type_ids', torch.zeros_like(input_ids)).to(DEVICE)
                with torch.cuda.amp.autocast():
                    logits = model(input_ids, attention_mask, token_type_ids)
                all_probs.append(F.softmax(logits, dim=-1).cpu().numpy())

        oof_probs = np.vstack(all_probs)
        oof_preds = oof_probs.argmax(axis=-1)
        f1 = f1_score(va_labels, oof_preds, average='macro')
        avg_loss = total_loss / len(tr_loader)
        print(f'  Epoch {epoch} | loss={avg_loss:.4f} | val_f1={f1:.4f}')

        if f1 > best_f1:
            best_f1 = f1
            best_oof_probs = oof_probs.copy()
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f'  Best val F1: {best_f1:.4f}')

    # --- Test predictions using best model ---
    model.load_state_dict(best_state)
    model.eval()

    test_enc = tokenizer(test_df['report'].tolist(), truncation=True, padding='max_length',
                         max_length=MAX_LEN, return_tensors='pt')
    test_ds  = ReportDataset(test_enc)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    test_probs = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            token_type_ids = batch.get('token_type_ids', torch.zeros_like(input_ids)).to(DEVICE)
            with torch.cuda.amp.autocast():
                logits = model(input_ids, attention_mask, token_type_ids)
            test_probs.append(F.softmax(logits, dim=-1).cpu().numpy())

    test_probs = np.vstack(test_probs)

    del model, optimizer, scheduler, scaler, fgm, criterion
    gc.collect()
    torch.cuda.empty_cache()

    return best_oof_probs, test_probs, best_f1

In [ ]:
# === Run BERT CV ===
gkf = GroupKFold(n_splits=N_SPLITS)
splits = list(gkf.split(train_df, labels, groups))

bert_oof  = np.zeros((len(train_df), NUM_LABELS))
bert_test = np.zeros((len(test_df), NUM_LABELS))
bert_scores = []

for fold, (tr_idx, va_idx) in enumerate(splits):
    oof_probs, test_probs, f1 = train_bert_fold(tr_idx, va_idx, fold)
    bert_oof[va_idx] = oof_probs
    bert_test += test_probs / N_SPLITS
    bert_scores.append(f1)

bert_oof_f1 = f1_score(labels, bert_oof.argmax(axis=1), average='macro')
print(f'\nBERT OOF F1: {bert_oof_f1:.4f}')
print(f'BERT Mean Fold F1: {np.mean(bert_scores):.4f} ± {np.std(bert_scores):.4f}')

---
## Level 0 — Model B: TF-IDF + CalibratedLinearSVC

In [ ]:
# === TF-IDF Vectorizers ===
tfidf_word = TfidfVectorizer(
    ngram_range=(1, 3), min_df=2, max_df=0.95,
    max_features=100_000, sublinear_tf=True,
)
tfidf_char = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 6), min_df=2, max_df=0.95,
    max_features=100_000, sublinear_tf=True,
)

train_texts = train_df['clean'].values
test_texts  = test_df['clean'].values

X_train_word = tfidf_word.fit_transform(train_texts)
X_test_word  = tfidf_word.transform(test_texts)

X_train_char = tfidf_char.fit_transform(train_texts)
X_test_char  = tfidf_char.transform(test_texts)

X_train_tfidf = hstack([X_train_word, X_train_char])
X_test_tfidf  = hstack([X_test_word, X_test_char])

print(f'TF-IDF features: word={X_train_word.shape[1]} + char={X_train_char.shape[1]} = {X_train_tfidf.shape[1]}')

In [ ]:
# === SVC CV ===
svc_oof  = np.zeros((len(train_df), NUM_LABELS))
svc_test = np.zeros((len(test_df), NUM_LABELS))
svc_scores = []

for fold, (tr_idx, va_idx) in enumerate(splits):
    print(f'\nSVC Fold {fold}')
    base_svc = LinearSVC(C=1.0, class_weight='balanced', max_iter=10000, random_state=SEED)
    cal_svc  = CalibratedClassifierCV(base_svc, cv=3, method='sigmoid')
    cal_svc.fit(X_train_tfidf[tr_idx], labels[tr_idx])

    oof_probs = cal_svc.predict_proba(X_train_tfidf[va_idx])
    svc_oof[va_idx] = oof_probs
    svc_test += cal_svc.predict_proba(X_test_tfidf) / N_SPLITS

    f1 = f1_score(labels[va_idx], oof_probs.argmax(axis=1), average='macro')
    svc_scores.append(f1)
    print(f'  F1: {f1:.4f}')

svc_oof_f1 = f1_score(labels, svc_oof.argmax(axis=1), average='macro')
print(f'\nSVC OOF F1: {svc_oof_f1:.4f}')
print(f'SVC Mean Fold F1: {np.mean(svc_scores):.4f} ± {np.std(svc_scores):.4f}')

---
## Level 0 — Model C: LightGBM with TF-IDF + Clinical Features

In [ ]:
# === LightGBM Features: TF-IDF (truncated via SVD) + Clinical ===
from sklearn.decomposition import TruncatedSVD

# Reduce TF-IDF to 200 dims for LGB (sparse doesn't work well with tree models)
svd = TruncatedSVD(n_components=200, random_state=SEED)
X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd  = svd.transform(X_test_tfidf)
print(f'SVD explained variance: {svd.explained_variance_ratio_.sum():.3f}')

# Combine SVD + Clinical
scaler = StandardScaler()
train_clin_scaled = scaler.fit_transform(train_clinical[CLINICAL_COLS].values)
test_clin_scaled  = scaler.transform(test_clinical[CLINICAL_COLS].values)

X_train_lgb = np.hstack([X_train_svd, train_clin_scaled])
X_test_lgb  = np.hstack([X_test_svd, test_clin_scaled])
print(f'LGB features: SVD={X_train_svd.shape[1]} + Clinical={len(CLINICAL_COLS)} = {X_train_lgb.shape[1]}')

In [ ]:
# === LightGBM CV ===
lgb_oof  = np.zeros((len(train_df), NUM_LABELS))
lgb_test = np.zeros((len(test_df), NUM_LABELS))
lgb_scores = []

for fold, (tr_idx, va_idx) in enumerate(splits):
    print(f'\nLGB Fold {fold}')
    dtrain = lgb.Dataset(X_train_lgb[tr_idx], label=labels[tr_idx])
    dvalid = lgb.Dataset(X_train_lgb[va_idx], label=labels[va_idx], reference=dtrain)

    model = lgb.train(
        LGB_PARAMS, dtrain, num_boost_round=LGB_ROUNDS,
        valid_sets=[dvalid], callbacks=[
            lgb.early_stopping(LGB_EARLY, verbose=False),
            lgb.log_evaluation(200),
        ],
    )

    oof_probs = model.predict(X_train_lgb[va_idx])
    lgb_oof[va_idx] = oof_probs
    lgb_test += model.predict(X_test_lgb) / N_SPLITS

    f1 = f1_score(labels[va_idx], oof_probs.argmax(axis=1), average='macro')
    lgb_scores.append(f1)
    print(f'  F1: {f1:.4f}')

lgb_oof_f1 = f1_score(labels, lgb_oof.argmax(axis=1), average='macro')
print(f'\nLGB OOF F1: {lgb_oof_f1:.4f}')
print(f'LGB Mean Fold F1: {np.mean(lgb_scores):.4f} ± {np.std(lgb_scores):.4f}')

---
## Level 1 — Stacking Meta-Learner

In [ ]:
# === Combine OOF Probabilities from all 3 models ===
print('=== Individual Model OOF Results ===')
print(f'  BERT OOF F1:  {bert_oof_f1:.4f}')
print(f'  SVC  OOF F1:  {svc_oof_f1:.4f}')
print(f'  LGB  OOF F1:  {lgb_oof_f1:.4f}')

# Stack: 7*3 = 21 probability features
X_stack_train = np.hstack([bert_oof, svc_oof, lgb_oof])
X_stack_test  = np.hstack([bert_test, svc_test, lgb_test])
print(f'\nStacking features: {X_stack_train.shape[1]}')

In [ ]:
# === Simple Weighted Average Baseline ===
# Grid search best weights on OOF
best_w_f1 = 0
best_weights = (0.7, 0.15, 0.15)

for wb in np.arange(0.40, 0.91, 0.05):
    for ws in np.arange(0.05, 0.51, 0.05):
        wl = 1.0 - wb - ws
        if wl < 0.05:
            continue
        ens_probs = wb * bert_oof + ws * svc_oof + wl * lgb_oof
        f1 = f1_score(labels, ens_probs.argmax(axis=1), average='macro')
        if f1 > best_w_f1:
            best_w_f1 = f1
            best_weights = (wb, ws, wl)

print(f'Best weighted avg: F1={best_w_f1:.4f} | w_bert={best_weights[0]:.2f}, w_svc={best_weights[1]:.2f}, w_lgb={best_weights[2]:.2f}')

In [ ]:
# === Stacking Meta-Learner (Ridge on OOF probs) ===
# This learns optimal combination + can capture non-linear interactions between model outputs
from sklearn.linear_model import RidgeClassifierCV

meta_oof_preds = np.zeros(len(train_df), dtype=int)
meta_test_preds_all = []

for fold, (tr_idx, va_idx) in enumerate(splits):
    meta = RidgeClassifierCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], class_weight='balanced')
    meta.fit(X_stack_train[tr_idx], labels[tr_idx])
    meta_oof_preds[va_idx] = meta.predict(X_stack_train[va_idx])
    meta_test_preds_all.append(meta.decision_function(X_stack_test))

meta_f1 = f1_score(labels, meta_oof_preds, average='macro')
print(f'Stacking (Ridge) OOF F1: {meta_f1:.4f}')

# Average test decision functions across folds
meta_test_avg = np.mean(meta_test_preds_all, axis=0)
meta_test_preds = meta_test_avg.argmax(axis=1)

In [ ]:
# === Choose best approach ===
print('\n=== FINAL COMPARISON ===')
print(f'  BERT only:       {bert_oof_f1:.4f}')
print(f'  SVC only:        {svc_oof_f1:.4f}')
print(f'  LGB only:        {lgb_oof_f1:.4f}')
print(f'  Weighted avg:    {best_w_f1:.4f}')
print(f'  Stacking (Ridge):{meta_f1:.4f}')

# Use whichever is best
if meta_f1 >= best_w_f1:
    print('\n>>> Using STACKING predictions')
    final_preds = meta_test_preds
else:
    print('\n>>> Using WEIGHTED AVERAGE predictions')
    wb, ws, wl = best_weights
    final_probs = wb * bert_test + ws * svc_test + wl * lgb_test
    final_preds = final_probs.argmax(axis=1)

In [ ]:
# === Generate Submission ===
sub = pd.DataFrame({'ID': test_df['ID'], 'target': final_preds})
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(sub['target'].value_counts().sort_index())
print(f'\nSubmission saved: {sub.shape}')
sub.head()